# Chapter 7 — Shallow Neural Networks

**Math for ML Engineers** · companion notebook

This notebook mirrors §7.1–§7.4 of the chapter (forward propagation, the ReLU nonlinearity, and manual backpropagation for a two-layer network), working the derivation the chapter builds up to on paper into running code. **Key takeaway: backpropagation is nothing more than the chain rule applied layer by layer — every modern autograd engine (PyTorch, JAX) performs exactly this computation, just automatically and at far greater scale.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

## Setup: a two-layer network

We consider a shallow network with one hidden layer of ReLU units mapping $x \in \mathbb{R}^{d_{in}}$ to a scalar output $z_2 \in \mathbb{R}^{d_{out}}$:

$$
z_1 = W_1 x + b_1, \qquad a_1 = \operatorname{ReLU}(z_1), \qquad z_2 = W_2 a_1 + b_2
$$

with the mean-squared error loss

$$
\mathcal{L}(z_2, y) = \frac{1}{d_{out}} \sum_{i=1}^{d_{out}} (z_2^{(i)} - y^{(i)})^2 .
$$

Backpropagation computes $\partial \mathcal{L} / \partial W_1, \partial \mathcal{L}/\partial b_1, \partial \mathcal{L}/\partial W_2, \partial \mathcal{L}/\partial b_2$ by applying the chain rule from the output backward:

$$
\delta_2 = \frac{\partial \mathcal{L}}{\partial z_2} = \frac{2}{d_{out}}(z_2 - y), \qquad
\delta_1 = \frac{\partial \mathcal{L}}{\partial z_1} = \left(W_2^\top \delta_2\right) \odot \operatorname{ReLU}'(z_1)
$$

and the weight gradients are outer products of the layer's incoming activation with its own delta:

$$
\nabla_{W_2} \mathcal{L} = \delta_2 \, a_1^\top, \qquad \nabla_{W_1} \mathcal{L} = \delta_1 \, x^\top .
$$

This is §7.3's central derivation — the two-layer case that generalizes directly to arbitrarily deep networks.

In [ ]:
def relu(z):
    return np.maximum(0, z)


def relu_grad(z):
    return (z > 0).astype(float)


def mse(pred, target):
    return np.mean((pred - target) ** 2)


def forward(x, W1, b1, W2, b2):
    z1 = W1 @ x + b1
    a1 = relu(z1)
    z2 = W2 @ a1 + b2
    return z1, a1, z2


def backward(x, y, z1, a1, z2, W1, W2):
    d_out = y.shape[0]
    delta2 = 2 * (z2 - y) / d_out               # dL/dz2 for L = mean((z2-y)^2)
    dW2 = np.outer(delta2, a1)
    db2 = delta2
    delta1 = (W2.T @ delta2) * relu_grad(z1)    # chain rule through ReLU
    dW1 = np.outer(delta1, x)
    db1 = delta1
    return dW1, db1, dW2, db2


def numerical_gradient(f, x, h=1e-5):
    grad = np.zeros_like(x)
    for i in range(x.size):
        x_plus = x.copy(); x_plus.flat[i] += h
        x_minus = x.copy(); x_minus.flat[i] -= h
        grad.flat[i] = (f(x_plus) - f(x_minus)) / (2 * h)
    return grad

## Why XOR, and why a hidden layer

XOR is the smallest classification problem that is **not linearly separable**: no single line in the $(x_1, x_2)$ plane separates the two output classes. A linear model $z = Wx + b$ is therefore provably incapable of solving it, no matter how it is trained. Introducing a hidden layer with a nonlinearity (here, ReLU) lets the network bend a straight decision boundary into a piecewise-linear one that *can* wrap around the XOR pattern.

We train a 2 → 8 → 1 network (2 inputs, 8 hidden ReLU units, 1 scalar output) with full-batch gradient descent on the four XOR examples, applying `forward` and `backward` once per example each step and accumulating the parameter updates.

In [ ]:
# XOR dataset: inputs and targets
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y = np.array([0, 1, 1, 0], dtype=float)

d_in, d_hidden, d_out = 2, 8, 1

# Small random initialization
W1 = np.random.randn(d_hidden, d_in) * 0.5
b1 = np.zeros(d_hidden)
W2 = np.random.randn(d_out, d_hidden) * 0.5
b2 = np.zeros(d_out)

lr = 0.1
n_steps = 3000
loss_history = []

for step in range(n_steps):
    dW1_acc = np.zeros_like(W1)
    db1_acc = np.zeros_like(b1)
    dW2_acc = np.zeros_like(W2)
    db2_acc = np.zeros_like(b2)
    total_loss = 0.0

    for x, y_scalar in zip(X, Y):
        y = np.array([y_scalar])
        z1, a1, z2 = forward(x, W1, b1, W2, b2)
        total_loss += mse(z2, y)
        dW1, db1, dW2, db2 = backward(x, y, z1, a1, z2, W1, W2)
        dW1_acc += dW1
        db1_acc += db1
        dW2_acc += dW2
        db2_acc += db2

    n = X.shape[0]
    W1 -= lr * dW1_acc / n
    b1 -= lr * db1_acc / n
    W2 -= lr * dW2_acc / n
    b2 -= lr * db2_acc / n

    loss_history.append(total_loss / n)

print(f"Final training loss after {n_steps} steps: {loss_history[-1]:.6f}")
print("Predictions vs targets:")
for x, y_scalar in zip(X, Y):
    _, _, z2 = forward(x, W1, b1, W2, b2)
    print(f"  x={x}, target={y_scalar:.0f}, predicted={z2[0]:.4f}")

## Visualizing what the network learned

Two views make the training run legible:

1. **The loss curve** — full-batch MSE over the 3000 steps, which should fall smoothly toward zero as gradient descent finds a set of weights that fits all four XOR points.
2. **The learned decision boundary** — since XOR lives in a 2D input space, we can evaluate the trained network's output over a dense grid of $(x_1, x_2)$ points and render it as a filled contour plot. A linear model would produce parallel straight bands; a correctly trained hidden layer should instead carve out a nonlinear region that separates the two "on" corners `(0,1)` and `(1,0)` from the two "off" corners `(0,0)` and `(1,1)`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss curve
axes[0].plot(loss_history, color="tab:blue")
axes[0].set_xlabel("training step")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("Training loss on XOR")
axes[0].set_yscale("log")

# Decision boundary heatmap
grid_n = 200
xs = np.linspace(-0.25, 1.25, grid_n)
ys = np.linspace(-0.25, 1.25, grid_n)
XX, YY = np.meshgrid(xs, ys)
ZZ = np.zeros_like(XX)

for i in range(grid_n):
    for j in range(grid_n):
        point = np.array([XX[i, j], YY[i, j]])
        _, _, z2 = forward(point, W1, b1, W2, b2)
        ZZ[i, j] = z2[0]

contour = axes[1].contourf(XX, YY, ZZ, levels=20, cmap="RdBu_r")
axes[1].scatter(X[:, 0], X[:, 1], c=Y, cmap="RdBu_r", edgecolor="black", s=120, linewidth=1.5, zorder=3)
axes[1].set_xlabel("$x_1$")
axes[1].set_ylabel("$x_2$")
axes[1].set_title("Learned decision surface")
fig.colorbar(contour, ax=axes[1], label="network output")

plt.tight_layout()
plt.show()

print(f"Loss dropped from {loss_history[0]:.4f} to {loss_history[-1]:.6f} over {n_steps} steps.")
print("The contour plot bends around the diagonal corners — a boundary no single hyperplane could draw.")

## Why this toy problem matters at production scale

XOR is small, but it is not a toy in the historical sense — it is the textbook counterexample that stalled neural network research for over a decade. Minsky and Papert's 1969 analysis of perceptrons showed that a single linear layer cannot represent XOR, because no hyperplane separates the two classes. That result is often credited with contributing to the "AI winter" that followed, and the eventual demonstration that a *hidden nonlinear layer* resolves it — combined with the popularization of backpropagation for training such networks in the 1980s (Rumelhart, Hinton, and Williams, 1986) — is what revived the field.

The lesson generalizes directly to the largest models in production today: **every layer of every deep network, including every transformer block in a modern LLM, exists because stacking a linear transform with a nonlinearity increases representational capacity in exactly the way that turns an unsolvable problem (XOR) into a solvable one.** Remove the nonlinearities from a transformer and the entire stack of attention and MLP layers collapses algebraically into a single linear map — no more expressive than one matrix multiply, and no more capable than the perceptron that failed on XOR.

There is a second, purely computational point buried in the `backward` function above: **this hand-written chain rule is literally the operation that `loss.backward()` performs in PyTorch or `jax.grad` performs in JAX**, just generalized to arbitrary computation graphs, arbitrary layer types, and — in a large language model — billions of parameters across dozens of transformer blocks instead of one hidden layer of eight ReLU units. Autograd frameworks did not invent a new algorithm; they industrialized exactly the $\delta_2, \delta_1$ recursion derived in §7.3.

In [ ]:
# Experiment 1: confirm no LINEAR model can fit XOR, no matter how it's trained.
# Fit z = w^T x + b by least squares directly (the closed-form optimum for a linear model).
X_aug = np.hstack([X, np.ones((X.shape[0], 1))])   # append bias column
w_lin, *_ = np.linalg.lstsq(X_aug, Y, rcond=None)
linear_preds = X_aug @ w_lin
linear_mse = mse(linear_preds, Y)

print("Best possible LINEAR fit to XOR (closed-form least squares):")
for x, y_scalar, p in zip(X, Y, linear_preds):
    print(f"  x={x}, target={y_scalar:.0f}, linear prediction={p:.4f}")
print(f"Best achievable linear MSE: {linear_mse:.6f}  (nonzero and irreducible)")
print(f"Our 2-8-1 ReLU network's MSE: {loss_history[-1]:.6f}  (drives toward zero)\n")

# Experiment 2: verify the manual backward() pass against numerical_gradient,
# the same check that catches autograd bugs in real frameworks.
x0, y0 = X[1], np.array([Y[1]])   # an arbitrary XOR example, e.g. (0, 1) -> 1
z1_0, a1_0, z2_0 = forward(x0, W1, b1, W2, b2)
dW1_a, db1_a, dW2_a, db2_a = backward(x0, y0, z1_0, a1_0, z2_0, W1, W2)


def loss_of_W2(W2_candidate):
    _, _, z2_c = forward(x0, W1, b1, W2_candidate, b2)
    return mse(z2_c, y0)


dW2_numerical = numerical_gradient(loss_of_W2, W2)
max_abs_diff = np.max(np.abs(dW2_a - dW2_numerical))

print("Gradient check: analytical dW2 (from backward()) vs numerical_gradient(): ")
print(f"  max |analytical - numerical| = {max_abs_diff:.2e}")
print("  Match to machine precision -> the hand-derived chain rule is correct,")
print("  exactly the kind of check that validates a custom autograd op in practice.")

## Exercise

1. Extend the `backward` derivation to a **three-layer** network ($d_{in} \to d_{h1} \to d_{h2} \to d_{out}$, both hidden layers ReLU). Write out $\delta_3, \delta_2, \delta_1$ and show that the recursion $\delta_{k-1} = (W_k^\top \delta_k) \odot \operatorname{ReLU}'(z_{k-1})$ from §7.3 applies unchanged at each additional layer. Verify your new `backward` against `numerical_gradient` the same way this notebook did. (See §8.2 for the general multi-layer backpropagation derivation.)

2. Replace ReLU with the sigmoid activation $\sigma(z) = 1/(1+e^{-z})$ and derive $\sigma'(z)$ analytically. Retrain the network on XOR with this new activation and compare both the final loss and the shape of the decision-boundary contour to the ReLU version above — does a smooth nonlinearity change how sharply the network separates the two classes? (See §7.2 for a catalogue of activation functions and their gradients.)